# Project Pipeline

Stage 06. **I based it on the lecture notebook.**

## Install Missing Packages

In [1]:
# Install missing packages (uncomment and run to install).
# !pip install pandas python-dotenv pyarrow

## Load and Config Environment

In [2]:
from pathlib import Path

# Project root.
ROOT = Path.cwd()
if not (ROOT/".env.example").exists() and (ROOT.parent/".env.example").exists():
    ROOT = ROOT.parent

CHECKS = [
    ("src/config.py", "NEEDED", "env helpers"),
    ("src/utils.py", "NEEDED", "summary stats"),
    ("src/io_utils.py", "NEEDED", "save and load helpers"),
    ("src/cleaning.py", "NEEDED", "Stage 06 cleaning helpers"),
    ("data/raw/prismatic_evoluations_prices.csv", "NEEDED", "Prismatic Evolutions panel"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT/rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    raise FileNotFoundError(f"{missing} needed file(s) missing under {ROOT}")
print("\nAll needed files present.")

Looking in: /Users/ghostof0days/projects/bootcamp/project

  [OK ]  NEEDED    src/config.py                       env helpers
  [OK ]  NEEDED    src/utils.py                        summary stats
  [OK ]  NEEDED    src/io_utils.py                     save and load helpers
  [OK ]  NEEDED    src/cleaning.py                     Stage 06 cleaning helpers
  [OK ]  NEEDED    data/raw/prismatic_evoluations_prices.csv  Prismatic Evolutions panel

All needed files present.


In [3]:
import sys

import pandas as pd
from dotenv import load_dotenv

# Import helpers from src/.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cleaning import drop_missing, fill_missing_median, normalize_data
from src.config import get_key
from src.io_utils import read_df, validate, write_df
from src.utils import get_summary_stats

# Load `.env`.
load_dotenv(ROOT/".env")

RAW = ROOT/(get_key("DATA_DIR_RAW", "data/raw") or "data/raw")
PROC = ROOT/(get_key("DATA_DIR_PROCESSED", "data/processed") or "data/processed")
print("RAW:", RAW.resolve())
print("PROC:", PROC.resolve())

RAW -> /Users/ghostof0days/projects/bootcamp/project/data/raw
PROC -> /Users/ghostof0days/projects/bootcamp/project/data/processed


## Load, validate, and summarize CSV data

In [4]:
card_prices = pd.read_csv(RAW/"prismatic_evoluations_prices.csv", parse_dates=["date"])
print(validate(card_prices, ["date", "card_name", "market_price"]))
get_summary_stats(card_prices)

{'missing': [], 'shape': (80, 4), 'na_total': 0}
[2026-08-18 05:54:19.357305] Function 'get_summary_stats' called.


,market_price
count,80.000000
mean,42.040500
std,53.259212
min,0.140000
25%,0.320000
50%,9.090000
75%,80.220000
max,152.250000


## Clean and save CSV prices (fill in missing market prices as well as only take valid price). For this, I define valid as positive.

In [5]:
positive = card_prices[card_prices["market_price"] > 0].copy()
cleaned = normalize_data(
    drop_missing(fill_missing_median(positive, ["market_price"]), threshold=0.5),
    ["market_price"],
)
cleaned.to_csv(PROC/"prismatic_prices_cleaned.csv", index=False)
print("Cleaned shape:", cleaned.shape)

Cleaned shape: (80, 4)


## Save and load the cleaned prices as a Parquet file.

In [ ]:
parquet_path = PROC/"prismatic_prices_cleaned.parquet"
write_df(cleaned, parquet_path)
parquet_prices = read_df(parquet_path)
print("Parquet shape:", parquet_prices.shape)

Parquet shape: (80, 4)
